In [1]:
# Import Packages
import os
import re
import sys
import time
import copy
from pathlib import Path
from IPython.display import display

import numpy as np
import pandas as pd

from itertools import combinations
from itertools import product

from sklearn.utils.parallel import Parallel, delayed

%run Model_functions.ipynb

In [2]:
BASE_DIR = "../../DATA/AGB_DATA/Merged_Data/"
#BASE_DIR = "../../Data/"

In [3]:
def filter_existing(feat_list, available):
    """Keep only features that survived filtering. Warn about dropped ones."""
    kept    = [f for f in feat_list if f in available]
    dropped = [f for f in feat_list if f not in available]
    if dropped:
        print(f"  Dropped (not in dataframe): {dropped}")
    return kept

In [4]:
def create_interact_terms(X, interact_col_names):
    for col in interact_col_names:
        if col in X.columns:
            X[f'height_x_{col}'] = X['height'] * X[col]

    return X

In [5]:
def generate_feature_combinations(feature_groups, categ_cols, required_groups=None):
    if required_groups is None:
        required_groups = []

    combos = []
    group_names = list(feature_groups.keys())

    for r in range(1, len(group_names) + 1):
        for selected_groups in combinations(group_names, r):

            # Skip useless combos
            if all(g in categ_cols for g in selected_groups):
                continue
            
            features = []
            for group in selected_groups:
                features.extend(feature_groups[group])

            features = list(dict.fromkeys(features))

            combos.append({
                "group_combo": selected_groups,
                "features": features,
                "num_features": len(features)
            })

    return combos

# EMIT + CANOPY DATA

In [7]:
EMIT_DATA_CSV = BASE_DIR + "/EMIT_AGB/AGB_EMIT_EO_CANOPY.csv"

emit_raw_df = pd.read_csv(EMIT_DATA_CSV)
print(emit_raw_df.shape)

assert len(emit_raw_df["simard_height_m"].head())
assert len(emit_raw_df["tandemx_height_m"].head())

(3880, 316)


## Handle EMIT columns with NULL data

In [9]:
emit_raw_df = handle_null_columns(emit_raw_df)

Total NULL count           : 17604
Rows with at least one NULL: 1467
Total rows                 : 3880
Percentage                 : 37.8%

NULL count per column in affected rows:
EMIT_R1357    1467
EMIT_R1796    1467
EMIT_R1789    1467
EMIT_R1781    1467
EMIT_R1774    1467
EMIT_R1432    1467
EMIT_R1424    1467
EMIT_R1417    1467
EMIT_R1350    1467
EMIT_R1342    1467
EMIT_R1335    1467
EMIT_R1327    1467
dtype: int64
Dropping 12 columns:
['EMIT_R1327', 'EMIT_R1335', 'EMIT_R1342', 'EMIT_R1350', 'EMIT_R1357', 'EMIT_R1417', 'EMIT_R1424', 'EMIT_R1432', 'EMIT_R1774', 'EMIT_R1781', 'EMIT_R1789', 'EMIT_R1796']

NULL count after dropping: 0


## Remove Low Variance Features (cols)

A column that's nearly constant carries no predictive information.

In [11]:
emit_raw_df, low_var_removed = remove_low_variance_cols(emit_raw_df,
                                                        exclude_cols=['height'],
                                                        threshold=0.01,
                                                        exclude_categorical=True)
assert emit_raw_df is not None

print(f"Low variance columns removed: {low_var_removed}")

Total low variance columns removed: 99
Features after variance filtering: 304
Low variance columns removed: ['NDRE2', 'NDRE3', 'EMIT_R1477', 'EMIT_R1484', 'EMIT_R1491', 'EMIT_R1499', 'EMIT_R1506', 'EMIT_R1514', 'EMIT_R1521', 'EMIT_R1529', 'EMIT_R1536', 'EMIT_R1544', 'EMIT_R1551', 'EMIT_R1558', 'EMIT_R1566', 'EMIT_R1573', 'EMIT_R1581', 'EMIT_R1588', 'EMIT_R1596', 'EMIT_R1603', 'EMIT_R1611', 'EMIT_R1618', 'EMIT_R1625', 'EMIT_R1633', 'EMIT_R1640', 'EMIT_R1648', 'EMIT_R1655', 'EMIT_R1663', 'EMIT_R1670', 'EMIT_R1967', 'EMIT_R1975', 'EMIT_R1982', 'EMIT_R1990', 'EMIT_R1997', 'EMIT_R2004', 'EMIT_R2012', 'EMIT_R2019', 'EMIT_R2027', 'EMIT_R2034', 'EMIT_R2041', 'EMIT_R2049', 'EMIT_R2056', 'EMIT_R2064', 'EMIT_R2071', 'EMIT_R2079', 'EMIT_R2086', 'EMIT_R2093', 'EMIT_R2101', 'EMIT_R2108', 'EMIT_R2116', 'EMIT_R2123', 'EMIT_R2130', 'EMIT_R2138', 'EMIT_R2145', 'EMIT_R2153', 'EMIT_R2160', 'EMIT_R2167', 'EMIT_R2323', 'EMIT_R2330', 'EMIT_R2338', 'EMIT_R2345', 'EMIT_R2353', 'EMIT_R2360', 'EMIT_R2367', 'EMIT

In [12]:
available_cols = set(emit_raw_df.columns)

## Feature selection

In [14]:
emit_non_feature_cols = [
    'dataset',             # metadata
    'EMIT_selected_date',  # metadata
    'EMIT_granule',        # metadata
    'start_date',          # metadata
    'end_date',            # metadata
    'capture_start',       # metadata
    'capture_end',         # metadata
    'latitude',            # coordinate
    'longitude',           # coordinate
    'plant_AGB_kg',        # Target variable
    'diameter',            # Allometric
    'height'               # Allometric
]
emit_feature_cols = [c for c in emit_raw_df.columns if c not in emit_non_feature_cols]

target = 'plant_AGB_kg'
y_emit = emit_raw_df[target]

### Feature selection with Simard canopy height

In [16]:
X_emit_t = emit_raw_df[emit_feature_cols].copy()

emit_categ_cols = get_categorical_cols(X_emit_t)

# Select TANDEMX
X_emit_t = X_emit_t.rename({'tandemx_height_m': 'height'}, axis=1)
X_emit_t = X_emit_t.drop(columns=['simard_height_m'])

#### Create interaction terms

In [18]:
EMIT_interact_cols = filter_existing(['EVI', 'MSI', 'NBR', 'NDRE1', 'NDRE2', 'NDRE3', 'CIrededge'],
                                     available_cols)

  Dropped (not in dataframe): ['NDRE2', 'NDRE3']


In [19]:
X_emit_t = create_interact_terms(X_emit_t, EMIT_interact_cols)

### Feature selection with TandemX canopy height

In [21]:
X_emit_s = emit_raw_df[emit_feature_cols].copy()

# Select SIMARD
X_emit_s = X_emit_s.rename({'simard_height_m': 'height'}, axis=1)
X_emit_s = X_emit_s.drop(columns=['tandemx_height_m'])

#### Create interaction terms

In [23]:
X_emit_s = create_interact_terms(X_emit_s, EMIT_interact_cols)

## EMIT Test features

In [25]:
test_cv = 5

In [26]:
struct_features = ['height']
species         = ["species"]

emit_cols              = [c for c in X_emit_t.columns if c.startswith('EMIT')]
emit_indices           = filter_existing(['NDVI', 'NDRE1', 'NDRE2', 'NDRE3',
                                          'NBR', 'MSI', 'EVI', 'CIrededge'], available_cols)
emit_top_spectral      = filter_existing(['EVI', 'NBR', 'MSI', 'NDRE1'], available_cols)
emit_interaction_terms = [c for c in X_emit_t.columns if c.startswith('height_x_')]

emit_features_list = [
    # Non-spectral baselines
    struct_features,            # height only
    species,                    # species only
    struct_features + species,  # height + species (non-spectral floor)

    # Spectral alone
    emit_cols,                  # all 158 bands
    emit_indices,               # all 8 indices
    emit_top_spectral,          # 4 curated indices

    # Spectral + height
    emit_cols    + struct_features,
    emit_indices + struct_features,
    emit_top_spectral + struct_features,

    # Spectral + height + species
    emit_cols    + struct_features + species,
    emit_indices + struct_features + species,
    emit_top_spectral + struct_features + species,

    # Spectral + height + interactions
    emit_cols + struct_features + emit_interaction_terms,
    emit_top_spectral + struct_features + emit_interaction_terms,

    # Full spectrum of features
    emit_cols + struct_features + species + emit_interaction_terms,
    emit_top_spectral + struct_features + species + emit_interaction_terms,
]

  Dropped (not in dataframe): ['NDRE2', 'NDRE3']


## EMIT. Create plot groups.

In [28]:
# Retain the groups/plot_id for splitting the data based on groups.
plot_groups_emit = None
if 'plot_id' in X_emit_t.columns:
    plot_groups_emit = X_emit_t['plot_id'].copy()
    X_emit_t = X_emit_t.drop(columns=['plot_id'])

if 'plot_id' in X_emit_s.columns:
    X_emit_s = X_emit_s.drop(columns=['plot_id'])

near_zero_sites_emit, high_agb_sites_emit,\
    near_zero_plots_emit, high_agb_plots_emit = \
        get_low_and_high_agb_plots(y_emit,
                                   plot_groups_emit)

grp_emit = GROUP_INFO(near_zero_sites_emit,
                      high_agb_sites_emit,
                      near_zero_plots_emit,
                      high_agb_plots_emit,
                      groups=plot_groups_emit,
                      cv=test_cv)

assert plot_groups_emit is not None

High-AGB threshold  : 104.49 kg
Near-zero threshold : 0.004134

Near-zero variance plots:
  Big Creek_1               : log var = 0.000036
  Big Creek_4               : log var = 0.000032
  Frenchman Caye_1          : log var = 0.000753
  Frenchman Caye_2          : log var = 0.000381
  Frenchman Caye_3          : log var = 0.000693
  Frenchman Caye_4          : log var = 0.001306
  Frenchman Caye_5          : log var = 0.001283
  Frenchman Caye_6          : log var = 0.000158
  Shipstern Lagoon_1        : log var = 0.001064
  Shipstern Lagoon_3        : log var = 0.000232
  Shipstern Lagoon_4        : log var = 0.000113
  Shipstern Lagoon_5        : log var = 0.000052
  Shipstern Lagoon_6        : log var = 0.000135

High-AGB plots:
  Channel Caye_1            : max AGB = 310.9 kg
  Channel Caye_2            : max AGB = 206.4 kg
  Channel Caye_3            : max AGB = 427.2 kg
  Channel Caye_4            : max AGB = 237.6 kg
  Channel Caye_5            : max AGB = 170.4 kg
  Channel C

# EMIT EXPERIMENTS

In [30]:
test_cv = 5

In [31]:
LINEAR_FULL    = ["linear", "ridge", "lasso", "elasticnet"]
LINEAR_NO_OLS  = ["ridge", "lasso", "elasticnet"]

data_combos= [("EMIT+TANDEMX", X_emit_t, y_emit, emit_features_list, grp_emit, LINEAR_NO_OLS),
              ("EMIT+SIMARD" , X_emit_s, y_emit, emit_features_list, grp_emit, LINEAR_NO_OLS),
             ]
model_list    = ["linear", "rf", "xgboost", "lgbm", "merf"]
is_grids  = [False, True]
is_groups = [True]

**Why are we dropping "Linear Regression" for EMIT processing?**  
EMIT provides 158 hyperspectral bands after filtering, with strong collinearity between adjacent wavelengths — neighboring bands measure overlapping portions of the spectrum and carry nearly identical information. Plain ordinary least squares regression has no mechanism to handle this redundancy. With more features than informative directions, the design matrix becomes ill-conditioned and the solver is forced to assign large, unstable coefficients that cancel each other out on training data but extrapolate wildly on unseen data.  

For EMIT we therefore use only the regularized linear variants (Ridge, Lasso, ElasticNet), which constrain coefficient magnitudes and remain numerically stable on high-dimensional collinear features. Plain OLS is retained for Sentinel-2 (9 bands, minimal collinearity) where it is well-posed and serves as a useful unregularized baseline.

In [ ]:
global_experiment_list = {}
experiments_by_category = {}

for (combo_name, X, y, features_list, grp, linear_variants), model, use_grid, use_groups in product(
    data_combos, model_list, is_grids, is_groups
):
    print(f"\n{'='*70}")
    print(f"DATA: {combo_name} | MODEL: {model} | GRID: {use_grid} | GROUPED: {use_groups}")
    print('='*70)

    #run_label = f"{combo_name}, {model}, grid_{'yes' if use_grid else 'no'}, GROUPED: {'yes' if use_groups else 'no'}"

    exp_result = run_experiment(
        X, y,
        is_groups       = use_groups,
        group_info      = grp,
        features_list   = features_list,
        model_type      = model,
        is_grid         = use_grid,
        is_stratified   = False,
        exp_total_list = global_experiment_list,
        exp_by_category=experiments_by_category,
        linear_variants = linear_variants,
        label           = combo_name
    )


DATA: EMIT+TANDEMX | MODEL: linear | GRID: False | GROUPED: True

[1/48]

 EXPERIMENT-1. EMIT+TANDEMX,Model: RIDGE REGRESSION, Grouping? Yes, Hypertuned? No, Features: ['height']
Test R²     : 0.2427
Test RMSE   : 4.02 kg
Train R² (log scale): 0.2549
Train R² (orig scale): -0.0952
Train RMSE  : 19.43 kg
Num rows    : 3100
Num Features: 1

 Cross-validation ---
CV R² mean: 0.1820
CV R² std : 0.1966
CV scores : [-0.027  0.554  0.089  0.134  0.16 ]

Grouped Cross-validation ---
Grouped CV R² mean: 0.2253
Grouped CV R² std : 0.2148
Grouped CV scores : [-0.108  0.081  0.109 -0.026  0.399  0.526  0.577  0.184  0.206  0.307]
 EVALUATION: EXPERIMENT-1. EMIT+TANDEMX,Model: RIDGE REGRESSION, Grouping? Yes, Hypertuned? No, Features: ['height']

Test set:
  R²   : 0.243
  RMSE : 4.02 kg

 ✅ Test R² is positive (0.243)

Regular CrossValidation:
  Mean   : 0.182
  Std    : 0.197
  Scores : [-0.027  0.554  0.089  0.134  0.16 ]
 ✅ CV mean is positive (0.182)

Grouped CrossValidation:
  Big Creek_5   

# FIND THE BEST EXPERIMENT SO FAR.

## SUMMARY OF EXPERIMENTS

In [ ]:
%run Model_functions.ipynb
best_results = filter_best_experiments(global_experiment_list)
tab_df = tabulate_results(best_results)

## BEST ONE

In [ ]:
%run Model_functions.ipynb
best_result = best_results[0][1]
print_experiment(best_result)